# ScentAI — Offline Pipeline (One-Time)

This notebook consolidates all one-time offline processing for the ScentAI project.
It is designed to run on **Google Colab with an A100 (80 GB) or Blackwell 6000 GPU**.

## What this does
1. **LLM Enrichment** — fills all empty enrichment columns (`vibe_sentence`, `formality`, `day_night`, etc.)
   for all ~35k rows in `data/vibescent_enriched.csv` using Qwen3-8B (local GPU, no API keys).
2. **Corpus Re-embedding** — embeds every row's `retrieval_text` with `Qwen3-VL-Embedding-8B`
   and saves the resulting matrix to `artifacts/qwen3vl_corpus/embeddings.npy`.

## When to run
Run this notebook **once per dataset change**. After the artifacts are committed to the repo,
the live demo notebook (`harsh_week5_qwen3vl.ipynb`) loads them directly — this notebook does not
need to run again unless the CSV changes.

## Workflow: keeping Colab in sync with local changes

1. Edit code locally → run `./sync.sh` → pushes to GitHub
2. In Colab: **run the SYNC cell** (fast git pull, no reinstall) to pick up the latest code
3. On a fresh runtime: run **Cell 1 (Environment Setup)** once, restart, then continue from Stage 2

> **Note:** Cell 1 clones the repo from GitHub directly — no zip upload required.
> Google Drive is still mounted for the `uv` package cache and output artifacts.

## Stage Map
| Stage | Cell | Description |
|-------|------|-------------|
| SYNC | SYNC cell | Pull latest code from GitHub after `./sync.sh` — no reinstall |
| 1 | Cell 1 | Environment setup — git clone, install deps. **Restart runtime after this cell.** |
| 2 | Cell 2 | Config — paths, Drive mount for outputs, HF token |
| 3 | Cell 3 | Load & inspect dataset, resume from checkpoint if available |
| 4 | Cell 4 | LLM enrichment (Qwen3-8B, vLLM native guided decoding, batch 16) |
| 5 | Cell 5 | Corpus re-embedding (Qwen3-VL-Embedding-8B, batch 64) |
| 6 | Cell 6 | Verify artifacts |
| 7 | Cell 7 | Next steps — copy outputs to repo, update settings |


In [ ]:
# SYNC — Force-pull latest code from GitHub with LFS bypass
# Run after ./sync.sh on your local machine. Skip on brand-new runtimes.
import subprocess, os, sys

REPO_DIR = '/content/vibescent'

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('[INFO] Repo not yet cloned — run Stage 1 first.')
else:
    print('SYNC: Fetching latest from GitHub...')
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'fetch', 'origin', 'main', '--force'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'git fetch failed: {r.stderr.strip()}')

    # Bypass LFS budget errors
    subprocess.run(['git', '-C', REPO_DIR, 'config', 'filter.lfs.smudge', 'cat'], capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'config', 'filter.lfs.required', 'false'], capture_output=True)

    print('SYNC: Resetting to origin/main...')
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'git reset failed: {r.stderr.strip()}')

    # Clear stale vibescents module cache so imports pick up new code
    to_delete = [m for m in sys.modules if m.startswith("vibescents")]
    for m in to_delete:
        del sys.modules[m]
    if to_delete:
        print(f'[OK] Cleared module cache: {", ".join(to_delete)}')

    commit = subprocess.run(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f'SYNC complete. Commit: {commit}')


In [ ]:
# Stage 1: Professional Environment Setup
import subprocess, os, sys, shutil

REPO_DIR = "/content/vibescent"
GITHUB_REPO = "https://github.com/GavinGongTech/VibeScent.git"

print("Step 1: Efficient Repo Sync...")
subprocess.run(["git", "config", "--global", "filter.lfs.smudge", "cat"])
subprocess.run(["git", "config", "--global", "filter.lfs.required", "false"])

if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("  Using existing repo. Syncing with origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    print("  No valid repo found. Performing fresh shallow clone...")
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth=1", GITHUB_REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print("Step 2: Installing dependencies...")
try:
    import uv
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv", "hf-transfer"], check=True)

# vLLM is required by VLLMNativeEnrichmentClient (Stage 4 enrichment)
print("  Installing vLLM...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
print("  vLLM ready.")

try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")
        print("  Autoreload enabled.")
except Exception as e:
    print(f"  [INFO] Could not enable autoreload: {e}")

print(f"Ready. Working Directory: {os.getcwd()}")
print("IMPORTANT: Restart the runtime now (Runtime -> Restart session).")

In [ ]:
# Stage 2: Config
import os, sys
REPO_DIR = '/content/vibescent'
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# Google Drive — all outputs saved here (survives session restart)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/vibescent_offline'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Paths
INPUT_CSV      = os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv')
ENRICHED_CSV   = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv')
CHECKPOINT_CSV = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv.ckpt')
FAILURES_LOG   = os.path.join(DRIVE_BASE, 'enrichment_failures.jsonl')
EMBEDDINGS_DIR = os.path.join(DRIVE_BASE, 'qwen3vl_corpus')
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

# INPUT_CSV is gitignored — copy from Drive if not present in the cloned repo.
# First-time setup: upload vibescent_enriched.csv to one of the Drive locations listed below.
# After the pipeline completes, Stage 7 copies the enriched CSV back here automatically.
if not os.path.exists(INPUT_CSV):
    import shutil as _shutil
    _drive_fallbacks = [
        os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv'),  # enriched output from a prior run
        '/content/drive/MyDrive/vibescent_enriched.csv',           # manual upload location
    ]
    for _src in _drive_fallbacks:
        if os.path.exists(_src):
            os.makedirs(os.path.dirname(INPUT_CSV), exist_ok=True)
            _shutil.copy(_src, INPUT_CSV)
            print(f'Copied INPUT_CSV from Drive: {_src}')
            break
    else:
        print(f'[ERROR] INPUT_CSV not found: {INPUT_CSV}')
        print( '        For first-time setup, upload vibescent_enriched.csv to one of:')
        for _c in _drive_fallbacks:
            print(f'          {_c}')
        raise FileNotFoundError('Input CSV missing. See instructions above.')
else:
    _size_mb = os.path.getsize(INPUT_CSV) // (1024 * 1024)
    print(f'Input CSV found in repo: {INPUT_CSV}  ({_size_mb} MB)')

# HF model cache on Drive — prevents re-downloading on each new session
HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# HF token from Colab Secrets (sidebar → 🔑 Secrets → + New secret: HF_TOKEN)
try:
    from google.colab import userdata as _ud
    _hf_token = _ud.get('HF_TOKEN') or ''
except Exception:
    _hf_token = os.environ.get('HF_TOKEN', '')

if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab Secrets.')
else:
    print('[WARN] HF_TOKEN not set — add via Colab sidebar → Secrets → HF_TOKEN')
    print('       Model download will fail without it.')

print(f'Config ready. Drive base: {DRIVE_BASE}')

In [ ]:
# Stage 3: Load and Inspect Dataset
import os, sys, json
import pandas as pd, numpy as np

REPO_DIR       = globals().get("REPO_DIR",       "/content/vibescent")
DRIVE_BASE     = globals().get("DRIVE_BASE",     "/content/drive/MyDrive/vibescent_offline")
INPUT_CSV      = globals().get("INPUT_CSV",       os.path.join(REPO_DIR, "data", "vibescent_enriched.csv"))
CHECKPOINT_CSV = globals().get("CHECKPOINT_CSV", os.path.join(DRIVE_BASE, "vibescent_enriched_full.csv.ckpt"))

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from vibescents.enrich import ENRICHMENT_COLUMNS

df_work = pd.read_csv(INPUT_CSV)
print(f"Loaded raw: {df_work.shape}")

# Resume from CSV checkpoint (written by Stage 4 every CHECKPOINT_EVERY rows)
if os.path.exists(CHECKPOINT_CSV):
    df_ckpt = pd.read_csv(CHECKPOINT_CSV)
    print(f"Applying CSV checkpoint: {CHECKPOINT_CSV}  ({df_ckpt.shape})")
    for col in ENRICHMENT_COLUMNS:
        if col in df_ckpt.columns:
            if col not in df_work.columns:
                df_work[col] = None
            mask = df_ckpt[col].notna()
            if mask.any():
                df_work.loc[mask[mask].index, col] = df_ckpt.loc[mask, col]
    done = df_work["vibe_sentence"].notna().sum() if "vibe_sentence" in df_work.columns else 0
    print(f"After merge: {done}/{len(df_work)} rows already enriched")
else:
    print("No checkpoint found — starting fresh.")

# Show enrichment coverage
for col in ENRICHMENT_COLUMNS:
    filled = df_work[col].notna().sum() if col in df_work.columns else 0
    pct = filled / len(df_work) * 100
    print(f"  {col}: {filled}/{len(df_work)} ({pct:.1f}%)")

if "fragrance_id" not in df_work.columns:
    df_work.insert(0, "fragrance_id", df_work.index.astype(str))
print(f"Ready. Shape: {df_work.shape}")


In [ ]:
# Stage 3b: Load Enrichment Client
# Colab A100/T4 GPU -> vLLM + Qwen3-8B  (~100 tok/sec, top-3788 rows in ~2 hrs)
# CPU / no GPU      -> Groq free API     (~14 rows/min, GROQ_API_KEY required)
import gc, os, subprocess, torch
from vibescents.enrich import VLLMNativeEnrichmentClient, GroqEnrichmentClient

ENRICHMENT_MODEL = "Qwen/Qwen3-8B"
GROQ_MODEL       = "llama-3.1-8b-instant"
BATCH_SIZE       = 16


def _vram_str() -> str:
    """Read GPU memory via nvidia-smi — works even when model is in a subprocess."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        used, total = [int(x.strip()) for x in out.split(",")]
        return f"{used/1024:.1f}/{total/1024:.1f} GB ({used*100//total}% used)"
    except Exception:
        return "n/a (nvidia-smi unavailable)"


if "client" in globals():
    print("Re-using existing client.")
    print(f"VRAM: {_vram_str()}")
else:
    gc.collect()
    _has_gpu = torch.cuda.is_available()
    print(f"GPU: {torch.cuda.get_device_name(0) if _has_gpu else 'not available'}")

    if _has_gpu:
        torch.cuda.empty_cache()
        try:
            print(f"Loading {ENRICHMENT_MODEL} via vLLM…")
            client = VLLMNativeEnrichmentClient(model_name=ENRICHMENT_MODEL)
            print(f"vLLM ready.  VRAM: {_vram_str()}")
            print(f"Speed: ~100 tok/s → top-3788 rows ≈ 2 hrs")
        except Exception as _e:
            print(f"vLLM failed: {_e}")
            _has_gpu = False

    if not _has_gpu:
        try:
            from google.colab import userdata as _ud
            _groq_key = _ud.get("GROQ_API_KEY") or os.environ.get("GROQ_API_KEY", "")
        except Exception:
            _groq_key = os.environ.get("GROQ_API_KEY", "")

        if not _groq_key:
            raise EnvironmentError(
                "No GPU found and GROQ_API_KEY not set.\n"
                "Option 1 (faster): Runtime → Change runtime type → T4/A100 GPU, then rerun\n"
                "Option 2 (free):   Sign up at https://console.groq.com\n"
                "                   Add secret GROQ_API_KEY via Colab sidebar → Secrets"
            )
        os.environ["GROQ_API_KEY"] = _groq_key
        client = GroqEnrichmentClient(model_name=GROQ_MODEL)
        ENRICHMENT_MODEL = GROQ_MODEL
        BATCH_SIZE = 1
        print(f"Groq ready. Model: {GROQ_MODEL}")
        print(f"Speed: ~14 rows/min (free tier). Top-3788 rows ≈ 4.5 hrs")

In [ ]:
# Stage 4: LLM Enrichment — sorted by popularity (most important rows first)
import os, sys, json, traceback
import torch, gc
import pandas as pd
from tqdm.auto import tqdm

REPO_DIR       = globals().get('REPO_DIR',       '/content/vibescent')
_drive_base    = '/content/drive/MyDrive/vibescent_offline'
CHECKPOINT_CSV = globals().get('CHECKPOINT_CSV',
    os.path.join(_drive_base, 'vibescent_enriched_full.csv.ckpt'))
ENRICHED_CSV   = globals().get('ENRICHED_CSV',
    os.path.join(_drive_base, 'vibescent_enriched_full.csv'))
FAILURES_LOG   = globals().get('FAILURES_LOG',
    os.path.join(_drive_base, 'enrichment_failures.jsonl'))
BATCH_SIZE = globals().get("BATCH_SIZE", 16)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from vibescents.enrich import (
    ENRICHMENT_COLUMNS, _build_prompt, _serialize_value, build_retrieval_text
)

assert "df_work" in globals(), "Run Stage 3 first to load df_work."
assert "client" in globals(), "Run Stage 3b first to load the enrichment client."

# Sort by popularity so top fragrances are enriched first (critical for 12-hr deadline)
if "rating_count" in df_work.columns:
    df_work["_sort"] = pd.to_numeric(df_work["rating_count"], errors="coerce").fillna(0)
    df_work = df_work.sort_values("_sort", ascending=False).drop(columns=["_sort"])
    df_work = df_work.reset_index(drop=True)
    print("Sorted by rating_count (most popular first)")

_need_mask = (
    df_work["vibe_sentence"].isna()
    if "vibe_sentence" in df_work.columns
    else pd.Series([True] * len(df_work), index=df_work.index)
)
_need_idx = df_work[_need_mask].index.tolist()
print(f"Rows needing enrichment: {len(_need_idx):,} / {len(df_work):,}")

# Show ETA
_eta_hrs = len(_need_idx) / 14 / 60
print(f"ETA: ~{_eta_hrs:.1f} hrs (Groq) or ~{len(_need_idx)*250/125/3600:.1f} hrs (Colab T4 vLLM)")

for col in ENRICHMENT_COLUMNS:
    if col not in df_work.columns:
        df_work[col] = None

CHECKPOINT_EVERY = 50
_completed, _failed = 0, 0
_failures = []

with tqdm(total=len(_need_idx), desc="Enriching") as pbar:
    for i in range(0, len(_need_idx), BATCH_SIZE):
        batch_idx = _need_idx[i:i+BATCH_SIZE]
        batch_prompts = [_build_prompt(df_work.loc[idx]) for idx in batch_idx]
        try:
            records = client.generate_batch(batch_prompts)
            for idx, record in zip(batch_idx, records):
                if record is None:
                    _failed += 1
                    _failures.append({"idx": idx, "error": "parse failed"})
                else:
                    for col in ENRICHMENT_COLUMNS:
                        df_work.at[idx, col] = _serialize_value(getattr(record, col))
                    _completed += 1
        except Exception as e:
            msg = f"Batch failed: {e}"
            for idx in batch_idx:
                _failed += 1
                _failures.append({"idx": idx, "error": msg})
        pbar.update(len(batch_idx))
        pbar.set_postfix(ok=_completed, fail=_failed)
        if (_completed + _failed) % CHECKPOINT_EVERY < BATCH_SIZE:
            df_work.to_csv(CHECKPOINT_CSV, index=False)

df_work.to_csv(CHECKPOINT_CSV, index=False)
df_work = build_retrieval_text(df_work)
df_work.to_csv(ENRICHED_CSV, index=False)

if _failures:
    with open(FAILURES_LOG, "w") as fh:
        for r in _failures: fh.write(json.dumps(r) + "\n")

print(f"\nEnrichment done: {_completed} ok, {_failed} failed")
print(f"Saved: {ENRICHED_CSV}")

# Release vLLM weights before Stage 5
try:
    if hasattr(client, '_llm'):
        client._llm = None
    del client
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"VRAM after release: {torch.cuda.memory_allocated()/1e9:.1f} GB")
except Exception as _e:
    print(f"[WARN] Could not release client: {_e}")

In [ ]:
# Stage 5: Corpus Embedding
# GPU path: Qwen3-VL-Embedding-8B (full quality, multimodal channel active)
# CPU path: nomic-embed-text-v1.5 (sentence-transformers, ~30s for 35k rows, loses multimodal)
import sys, os, time, gc, json, glob
import numpy as np
import pandas as pd
from pathlib import Path

REPO_DIR       = globals().get('REPO_DIR',       '/content/vibescent')
_drive_base    = '/content/drive/MyDrive/vibescent_offline'
ENRICHED_CSV   = globals().get('ENRICHED_CSV',   os.path.join(_drive_base, 'vibescent_enriched_full.csv'))
INPUT_CSV      = globals().get('INPUT_CSV',       os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'))
EMBEDDINGS_DIR = globals().get('EMBEDDINGS_DIR', os.path.join(_drive_base, 'qwen3vl_corpus'))
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

_csv = ENRICHED_CSV if os.path.exists(ENRICHED_CSV) else INPUT_CSV
df_embed = pd.read_csv(_csv)
print(f"Embedding {len(df_embed)} rows from: {_csv}")

if "retrieval_text" not in df_embed.columns or df_embed["retrieval_text"].isna().all():
    from vibescents.enrich import build_retrieval_text
    df_embed = build_retrieval_text(df_embed)

texts = df_embed["retrieval_text"].fillna(df_embed.get("name", "")).tolist()

# Try GPU embedder first; fall back to CPU sentence-transformers automatically
try:
    import torch
    gc.collect(); torch.cuda.empty_cache()
    torch.backends.cuda.matmul.allow_tf32 = True
    from vibescents.embeddings import Qwen3VLMultimodalEmbedder
    from vibescents.settings import Settings
    Qwen3VLMultimodalEmbedder._BATCH_SIZE = 64
    embedder = Qwen3VLMultimodalEmbedder(settings=Settings(), load_in_8bit=False)
    print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    _embed_fn = lambda chunk: embedder.embed_multimodal_documents(chunk)
    _mode = "GPU (Qwen3-VL-Embedding-8B, multimodal channel ACTIVE)"
except Exception as _gpu_err:
    print(f"GPU unavailable ({_gpu_err}), using CPU fallback...")
    from vibescents.embeddings import SentenceTransformerEmbedder
    embedder = SentenceTransformerEmbedder()
    _embed_fn = lambda chunk: embedder.embed_multimodal_documents(chunk, batch_size=256)
    _mode = "CPU (nomic-embed-text-v1.5, multimodal channel DISABLED)"

print(f"Mode: {_mode}")

out_emb      = os.path.join(EMBEDDINGS_DIR, "embeddings.npy")
out_manifest = os.path.join(EMBEDDINGS_DIR, "manifest.json")
CKPT_DIR     = os.path.join(EMBEDDINGS_DIR, "ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)

def get_checkpoints():
    return sorted(glob.glob(os.path.join(CKPT_DIR, "embeddings_ckpt_*.npy")),
                  key=lambda p: int(Path(p).stem.split("_")[-1]))

ckpt_files = get_checkpoints()
already_embedded = sum(np.load(f).shape[0] for f in ckpt_files)
print(f"Checkpoints: {len(ckpt_files)}, already done: {already_embedded}/{len(texts)}")

texts_to_embed = texts[already_embedded:]
if texts_to_embed:
    from tqdm.auto import tqdm
    CHUNK_SIZE = 3200  # larger chunks for CPU; GPU uses _BATCH_SIZE internally
    t0 = time.perf_counter()
    with tqdm(total=len(texts_to_embed), desc="Embedding") as pbar:
        for i in range(0, len(texts_to_embed), CHUNK_SIZE):
            chunk = texts_to_embed[i:i+CHUNK_SIZE]
            chunk_emb = _embed_fn(chunk)
            ckpt_path = os.path.join(CKPT_DIR, f"embeddings_ckpt_{len(get_checkpoints())}.npy")
            np.save(ckpt_path, chunk_emb)
            pbar.update(len(chunk))
    print(f"Embedded {len(texts_to_embed)} rows in {time.perf_counter()-t0:.0f}s")

all_ckpts = get_checkpoints()
embeddings = np.vstack([np.load(f) for f in all_ckpts]) if all_ckpts else np.empty((0,0), dtype=np.float32)
assert embeddings.shape[0] == len(texts), f"Shape mismatch: {embeddings.shape[0]} vs {len(texts)}"
assert np.isnan(embeddings).sum() == 0, "NaN in embeddings!"
print(f"Embeddings: {embeddings.shape}")

np.save(out_emb, embeddings)
manifest = {
    "model": _mode,
    "count": int(embeddings.shape[0]),
    "dim": int(embeddings.shape[1]),
    "l2_normalized": True,
    "source_csv": _csv,
}
with open(out_manifest, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved: {out_emb}")
print(f"Mode used: {_mode}")

# Stage 7: Next Steps — Copy Outputs to Repo

After all stages complete, copy the artifacts from Google Drive into the repo and commit:

```bash
# From repo root (or paste into a Colab shell cell with !)
mkdir -p artifacts/qwen3vl_corpus
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/embeddings.npy artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/manifest.json  artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/vibescent_enriched_full.csv   data/vibescent_enriched.csv
```

> **First-time setup:** `data/vibescent_enriched.csv` is gitignored and not tracked in the repo.
> Before running Cell 2, upload your raw `vibescent_enriched.csv` to Google Drive at one of:
> - `MyDrive/vibescent_offline/vibescent_enriched.csv`
> - `MyDrive/vibescent_enriched.csv`
>
> Cell 2 will copy it into the Colab session automatically.
> After the pipeline finishes, the `cp` command above writes the enriched CSV back to the repo
> so future runs can read it directly from the clone.

## `settings.py` — Already Updated

`corpus_embeddings_path` in `src/vibescents/settings.py` already points to the correct location:

```python
corpus_embeddings_path: str = str(DEFAULT_ARTIFACTS_DIR / 'qwen3vl_corpus' / 'embeddings.npy')
```

No changes to `settings.py` are needed.

## Run the Week 5 Demo Notebook Next

Open `notebooks/harsh_week5_qwen3vl.ipynb`.
**Skip Stage 0** (corpus re-embedding) — it is now complete.
Continue from Stage 1 (setup) as normal.